In [5]:
import numpy as np
import pandas as pd
from pathlib import Path
import zipfile
import requests
import logging

# === Paths ===
BASE_DIR    = Path(r"D:\Shikha\bias_project")
GLOVE_DIR   = BASE_DIR / "models" / "glove"
OUTPUT_DIR  = BASE_DIR / "outputs" / "embeddings" / "glove"
WORD_LISTS  = BASE_DIR / "word_lists" / "caliskan_2017"

GLOVE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# === GloVe file — using GloVe 6B 100d (small, fast, good for demo) ===
GLOVE_URL  = "https://nlp.stanford.edu/data/glove.6B.zip"
GLOVE_ZIP  = GLOVE_DIR / "glove.6B.zip"
GLOVE_FILE = GLOVE_DIR / "glove.6B.100d.txt"

# === Download if not already present ===
if GLOVE_FILE.exists():
    print(f"GloVe file already exists: {GLOVE_FILE}")
else:
    print("Downloading GloVe 6B vectors (~822MB)...")
    r = requests.get(GLOVE_URL, stream=True)
    total = int(r.headers.get('content-length', 0))
    downloaded = 0
    with open(GLOVE_ZIP, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            f.write(chunk)
            downloaded += len(chunk)
            pct = downloaded / total * 100 if total else 0
            print(f"\r  Downloaded {downloaded/1e6:.1f}MB ({pct:.1f}%)", end='')
    print("\nExtracting...")
    with zipfile.ZipFile(GLOVE_ZIP, 'r') as z:
        z.extract("glove.6B.100d.txt", GLOVE_DIR)
    print(f"Done. Saved to {GLOVE_FILE}")

print(f"\nGloVe file exists: {GLOVE_FILE.exists()}")
print(f"File size        : {GLOVE_FILE.stat().st_size / 1e6:.1f} MB")

GloVe file already exists: D:\Shikha\bias_project\models\glove\glove.6B.100d.txt

GloVe file exists: True
File size        : 347.1 MB


In [6]:
import requests
from pathlib import Path
import zipfile

GLOVE_DIR = BASE_DIR / "models" / "glove"
GLOVE_DIR.mkdir(parents=True, exist_ok=True)
GLOVE_ZIP = GLOVE_DIR / "glove.6B.zip"
GLOVE_FILE = GLOVE_DIR / "glove.6B.100d.txt"

GLOVE_URL = "https://nlp.stanford.edu/data/glove.6B.zip"

# Download with retries
for attempt in range(5):
    try:
        print(f"Download attempt {attempt+1}...")
        with requests.get(GLOVE_URL, stream=True, timeout=60) as r:
            r.raise_for_status()
            total = int(r.headers.get('content-length', 0))
            done = 0
            with open(GLOVE_ZIP, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    f.write(chunk)
                    done += len(chunk)
                    print(f"\r  {done/1e6:.0f}/{total/1e6:.0f} MB", end='')
        print("\nDownload complete.")
        break
    except Exception as e:
        print(f"\n  Failed: {e}")
        if attempt == 4:
            print("All attempts failed — check network.")

# Extract just the 100d file
if GLOVE_ZIP.exists():
    with zipfile.ZipFile(GLOVE_ZIP, 'r') as z:
        z.extract("glove.6B.100d.txt", GLOVE_DIR)
    print(f"Extracted. File exists: {GLOVE_FILE.exists()}")
    print(f"Size: {GLOVE_FILE.stat().st_size / 1e6:.1f} MB")

Download attempt 1...
  862/862 MB
Download complete.
Extracted. File exists: True
Size: 347.1 MB


In [7]:
# Load GloVe vectors into a dictionary
print("Loading GloVe vectors...")

glove = {}
with open(GLOVE_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.split()
        word = parts[0]
        vector = np.array(parts[1:], dtype=np.float32)
        glove[word] = vector

print(f"Loaded {len(glove):,} word vectors")
print(f"Vector dimension: {len(next(iter(glove.values())))}")

# === Load WEAT 6 word lists (same as Word2Vec notebook) ===
def load_word_list(filepath):
    with open(filepath, 'r') as f:
        return [w.strip().lower() for w in f.readlines() if w.strip()]

career_terms  = load_word_list(WORD_LISTS / "career_terms_weat6.txt")
family_terms  = load_word_list(WORD_LISTS / "family_terms_weat6.txt")
male_names    = load_word_list(WORD_LISTS / "male_names_weat6.txt")
female_names  = load_word_list(WORD_LISTS / "female_names_weat6.txt")

print(f"\nWEAT 6 word lists loaded:")
print(f"Career terms : {career_terms}")
print(f"Family terms : {family_terms}")
print(f"Male names   : {male_names}")
print(f"Female names : {female_names}")

# Verify all WEAT words are in GloVe vocabulary
print(f"\nVocabulary check:")
all_weat_words = career_terms + family_terms + male_names + female_names
missing = [w for w in all_weat_words if w not in glove]
print(f"WEAT words in GloVe : {len(all_weat_words) - len(missing)}/{len(all_weat_words)}")
if missing:
    print(f"Missing             : {missing}")
else:
    print("All WEAT words found in GloVe vocabulary!")

Loading GloVe vectors...
Loaded 400,000 word vectors
Vector dimension: 100

WEAT 6 word lists loaded:
Career terms : ['executive', 'management', 'professional', 'corporation', 'salary', 'office', 'business', 'career']
Family terms : ['home', 'parents', 'children', 'family', 'cousins', 'marriage', 'wedding', 'relatives']
Male names   : ['john', 'paul', 'mike', 'kevin', 'steve', 'greg', 'jeff', 'bill']
Female names : ['amy', 'joan', 'lisa', 'sarah', 'diana', 'kate', 'ann', 'donna']

Vocabulary check:
WEAT words in GloVe : 32/32
All WEAT words found in GloVe vocabulary!


In [8]:
import pandas as pd
import numpy as np
from gensim.utils import simple_preprocess

CORPUS_DIR = BASE_DIR / "corpus"
GARG_LISTS = BASE_DIR / "word_lists" / "garg_2018"

# Load gender pairs + attribute lists
male_pairs   = load_word_list(GARG_LISTS / "male_pairs.txt")
female_pairs = load_word_list(GARG_LISTS / "female_pairs.txt")

def tokenize(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return []
    return simple_preprocess(text, deacc=True, min_len=2)

# Build per-(state,period) vocab sets
cr_df       = pd.read_csv(CORPUS_DIR / "congressional_record" / "2024-01" / "speeches.csv")
ca_df       = pd.read_csv(CORPUS_DIR / "chronicling_america" / "1920-1922" / "articles.csv")
gd_df       = pd.read_csv(CORPUS_DIR / "gdelt" / "2024-01" / "gdelt_articles_with_text.csv")
gd_df       = gd_df[gd_df['text'].notna() & (gd_df['text'] != '')]
rd_posts    = pd.read_csv(CORPUS_DIR / "reddit" / "2020-01" / "posts.csv")
rd_comments = pd.read_csv(CORPUS_DIR / "reddit" / "2020-01" / "comments.csv")

EXCLUDE = {'GU','PR','VI','AS','MP'}
cell_vocab = {}  # (state, period) -> set of words present

def add_vocab(state, period, text):
    toks = tokenize(text)
    if len(toks) >= 5:
        cell_vocab.setdefault((state, period), set()).update(toks)

for _, r in ca_df.iterrows():
    add_vocab(str(r['state_abbr']), 'hist_1920s', str(r.get('text','')))
for _, r in rd_posts.iterrows():
    add_vocab(str(r['state_abbr']), 'modern_2020', str(r.get('title',''))+' '+str(r.get('selftext','')))
for _, r in rd_comments.iterrows():
    add_vocab(str(r['state_abbr']), 'modern_2020', str(r.get('body','')))
for _, r in cr_df.iterrows():
    add_vocab(str(r['state']), 'modern_2024', str(r.get('text','')))
for _, r in gd_df.iterrows():
    add_vocab(str(r['state']), 'modern_2024', str(r.get('text','')))

print(f"State-period cells: {len(cell_vocab)}")

State-period cells: 111


In [9]:
print(GLOVE_FILE.exists())
print(f"Size: {GLOVE_FILE.stat().st_size / 1e6:.1f} MB")

True
Size: 347.1 MB


In [10]:
import json

# === WEAT effect size using pretrained GloVe (A1: corpus-restricted coverage) ===
def cosine_sim(v1, v2):
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0:
        return 0.0
    return np.dot(v1, v2) / (n1 * n2)

def weat_effect_size_glove(vocab, X_words, Y_words, A_words, B_words,
                           min_targets=3, min_attrs=2):
    """
    Effect size using pretrained GloVe, but only counting words present
    in this cell's corpus vocab (period-specific coverage).
    Positive d = male (X) more associated with attribute A.
    """
    # Words must be BOTH in GloVe and in this cell's corpus
    def present(words):
        return [w for w in words if w in glove and w in vocab]

    Xp, Yp = present(X_words), present(Y_words)
    Ap, Bp = present(A_words), present(B_words)

    if len(Xp) < min_targets or len(Yp) < min_targets:
        return None, None
    if len(Ap) < min_attrs or len(Bp) < min_attrs:
        return None, None

    vecs_A = [glove[w] for w in Ap]
    vecs_B = [glove[w] for w in Bp]

    def s(word):
        wv = glove[word]
        return (np.mean([cosine_sim(wv, v) for v in vecs_A]) -
                np.mean([cosine_sim(wv, v) for v in vecs_B]))

    s_X = [s(w) for w in Xp]
    s_Y = [s(w) for w in Yp]
    all_s = s_X + s_Y
    std = np.std(all_s, ddof=1)
    if std == 0:
        return None, None
    d = (np.mean(s_X) - np.mean(s_Y)) / std
    cov = {'n_X': len(Xp), 'n_Y': len(Yp), 'n_A': len(Ap), 'n_B': len(Bp)}
    return d, cov

# Score all cells
results_glove = []
print(f"{'State':<7} {'Period':<13} {'WEAT6 d':>9} {'Coverage':>12}")
print("-" * 45)

for (state_abbr, period) in sorted(cell_vocab.keys()):
    if state_abbr in EXCLUDE:
        continue
    vocab = cell_vocab[(state_abbr, period)]
    d, cov = weat_effect_size_glove(vocab, male_pairs, female_pairs,
                                    career_terms, family_terms)
    if d is not None:
        cov_str = f"{cov['n_X']}/{cov['n_Y']}/{cov['n_A']}/{cov['n_B']}"
        print(f"{state_abbr:<7} {period:<13} {d:>9.4f} {cov_str:>12}")
        results_glove.append({
            'state': state_abbr, 'period': period,
            'weat6_d': round(d, 4),
            'cov_male': cov['n_X'], 'cov_female': cov['n_Y'],
            'cov_career': cov['n_A'], 'cov_family': cov['n_B'],
            'method': 'glove_pretrained_corpus_restricted'
        })

# Save
glove_df = pd.DataFrame(results_glove).sort_values(['period', 'state'])
OUT_GLOVE = OUTPUT_DIR / "weat6_scores_glove_v2.csv"
glove_df.to_csv(OUT_GLOVE, index=False)

print(f"\nCells scored: {len(glove_df)}")
for p in ['hist_1920s', 'modern_2020', 'modern_2024']:
    sub = glove_df[glove_df['period'] == p]
    if len(sub) > 0:
        print(f"  {p:<13}: {len(sub)} cells, mean d = {sub['weat6_d'].mean():.4f}")
print(f"Saved to: {OUT_GLOVE}")

State   Period          WEAT6 d     Coverage
---------------------------------------------
AK      modern_2020      1.0221      4/3/2/4
AK      modern_2024      1.1099    11/11/7/6
AL      modern_2020      0.5361      7/5/2/5
AL      modern_2024      0.9902      7/4/3/6
AR      modern_2020     -0.5978      7/4/2/2
AR      modern_2024      1.0200     8/10/4/3
AZ      modern_2024      0.3286      8/5/4/3
CA      hist_1920s       1.5537      8/4/2/3
CA      modern_2020      0.7506      4/3/4/2
CA      modern_2024      0.8709    12/13/5/7
CO      modern_2024      0.8490      9/6/3/4
CT      modern_2020      0.5131      6/3/3/4
CT      modern_2024      0.5200      9/8/3/3
DE      modern_2020     -0.0798      7/3/3/3
DE      modern_2024      0.9074      7/5/7/3
FL      modern_2020      1.1021      6/7/2/3
FL      modern_2024      0.8672    13/10/4/3
GA      modern_2020     -0.1401      8/3/4/3
GA      modern_2024      0.1797     13/9/5/3
HI      modern_2020      0.7221      6/5/4/4
HI      m